# 3) Categorical & Numerical Feature Analysis

This notebook analyzes the categorical and numerical features of the healthcare dataset before machine learning preprocessing.


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Load dataset directly from GitHub
url = "https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/main/healthcare-disease-prediction/dataset/healthcare_dataset.csv"
df = pd.read_csv(url)

print("Dataset loaded successfully.")
print(f"Dataset shape: {df.shape}")


## Categorical Feature Analysis

Categorical features contain labels, categories, names, or other textual information.

These features may require encoding before being used by machine learning algorithms.


In [ ]:
# Identify categorical/text columns
categorical_columns = df.select_dtypes(include="object").columns.tolist()

print("Categorical/Text Columns:\n")

for column in categorical_columns:
    print("-", column)

print(f"\nTotal categorical/text columns: {len(categorical_columns)}")


In [ ]:
# Display unique values and frequency counts for categorical columns

for column in categorical_columns:
    print(f"\n{column} - Unique Values: {df[column].nunique(dropna=True)}")
    print(df[column].value_counts(dropna=False).head(20))


In [ ]:
# Calculate cardinality of categorical features

categorical_summary = pd.DataFrame({
    "Column": categorical_columns,
    "Unique Values": [
        df[column].nunique(dropna=True)
        for column in categorical_columns
    ],
    "Unique Percentage": [
        (df[column].nunique(dropna=True) / len(df)) * 100
        for column in categorical_columns
    ]
})

categorical_summary


In [ ]:
# Identify low-cardinality categorical columns

low_cardinality_columns = [
    column
    for column in categorical_columns
    if df[column].nunique(dropna=True) <= 10
]

print("Low-Cardinality Categorical Columns:\n")

for column in low_cardinality_columns:
    print("-", column)

print(
    f"\nTotal low-cardinality categorical columns: "
    f"{len(low_cardinality_columns)}"
)


In [ ]:
# Plot distributions of low-cardinality categorical features

for column in low_cardinality_columns:
    plt.figure(figsize=(8, 4))

    sns.countplot(
        data=df,
        x=column,
        order=df[column].value_counts().index
    )

    plt.title(f"Distribution of {column}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## Numerical Feature Analysis

Numerical features contain values represented by numbers.

These features can be analyzed using descriptive statistics, distributions, skewness, and potential outliers.


In [ ]:
# Identify numerical columns

numerical_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

print("Numerical Columns:\n")

for column in numerical_columns:
    print("-", column)

print(f"\nTotal numerical columns: {len(numerical_columns)}")


In [ ]:
# Generate descriptive statistics for numerical features

numerical_summary = df[numerical_columns].describe().T

numerical_summary


In [ ]:
# Check skewness of numerical features

skewness = (
    df[numerical_columns]
    .skew(numeric_only=True)
    .sort_values(ascending=False)
)

skewness_df = pd.DataFrame({
    "Column": skewness.index,
    "Skewness": skewness.values
})

skewness_df


In [ ]:
# Plot distributions of numerical features

for column in numerical_columns:

    plt.figure(figsize=(8, 4))

    sns.histplot(
        data=df,
        x=column,
        kde=True
    )

    plt.title(f"Distribution of {column}")
    plt.tight_layout()
    plt.show()


In [ ]:
# Generate box plots for numerical features

for column in numerical_columns:

    plt.figure(figsize=(8, 4))

    sns.boxplot(
        data=df,
        x=column
    )

    plt.title(f"Box Plot of {column}")
    plt.tight_layout()
    plt.show()


In [ ]:
# Detect potential outliers using the IQR method

outlier_summary = []

for column in numerical_columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = (
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ).sum()

    outlier_summary.append({
        "Column": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Potential Outliers": outliers
    })

outlier_df = pd.DataFrame(outlier_summary)

outlier_df


## Target Variable Analysis

The target variable for the healthcare disease prediction project is:

**Medical Condition**

The distribution of the target classes is examined to determine whether the dataset is balanced.


In [ ]:
# Check target variable distribution

target_column = "Medical Condition"

print("Target Variable:", target_column)
print("\nTarget Classes:\n")

print(df[target_column].value_counts())


In [ ]:
# Calculate target class percentages

target_distribution = pd.DataFrame({
    "Count": df[target_column].value_counts(),
    "Percentage": (
        df[target_column]
        .value_counts(normalize=True) * 100
    )
})

target_distribution


In [ ]:
# Plot target variable distribution

plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x=target_column,
    order=df[target_column].value_counts().index
)

plt.title("Distribution of Medical Conditions")
plt.xlabel("Medical Condition")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Feature vs Target Analysis

Categorical features can be compared with the target variable to understand whether disease categories differ across feature groups.


In [ ]:
# Compare low-cardinality categorical features with the target

for column in low_cardinality_columns:

    if column != target_column:

        cross_tab = pd.crosstab(
            df[column],
            df[target_column],
            normalize="index"
        ) * 100

        print(f"\n{column} vs {target_column}")
        display(cross_tab.round(2))


In [ ]:
# Compare numerical features across target classes

for column in numerical_columns:

    plt.figure(figsize=(8, 5))

    sns.boxplot(
        data=df,
        x=target_column,
        y=column
    )

    plt.title(f"{column} by Medical Condition")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## Feature Analysis Summary

Categorical features are examined using:

- Unique-value counts
- Frequency distributions
- Cardinality
- Class distributions
- Feature-versus-target comparisons

Numerical features are examined using:

- Descriptive statistics
- Distribution plots
- Box plots
- Skewness
- Potential outlier detection
- Numerical feature versus target comparisons

No permanent feature removal or transformation is performed in this notebook.


In [ ]:
# Generate automatic feature analysis summary

print("FEATURE ANALYSIS SUMMARY")
print("=" * 50)

print(f"Total records           : {len(df)}")
print(f"Categorical/Text columns: {len(categorical_columns)}")
print(f"Numerical columns       : {len(numerical_columns)}")
print(
    f"Low-cardinality categorical columns: "
    f"{len(low_cardinality_columns)}"
)

print("\nCategorical columns:")

for column in categorical_columns:
    print(
        f"- {column}: "
        f"{df[column].nunique(dropna=True)} unique values"
    )

print("\nNumerical columns:")

for column in numerical_columns:
    print(f"- {column}")

print("\nTarget variable:")
print(f"- {target_column}")
print(
    f"- Number of classes: "
    f"{df[target_column].nunique(dropna=True)}"
)


# Conclusion

The categorical and numerical features of the healthcare dataset were analyzed to understand their distributions, cardinality, descriptive statistics, skewness, and potential outliers.

The target variable, **Medical Condition**, was also analyzed to understand the distribution of disease classes.

The findings from this notebook will support:

1. Exploratory Data Analysis
2. Initial data cleaning
3. Feature selection
4. Feature engineering
5. Machine learning preprocessing
